In [20]:
import warnings

warnings.filterwarnings('ignore')

In [21]:
### Установим красивые дефолтные настройки
### Может быть лень постоянно прописывать
### У графиков параметры цвета, размера, шрифта
### Можно положить их в словарь дефолтных настроек

import matplotlib as mlp

mlp.rcParams['lines.linewidth'] = 5
mlp.rcParams['xtick.major.size'] = 20
mlp.rcParams['xtick.major.width'] = 5
mlp.rcParams['xtick.labelsize'] = 20
mlp.rcParams['xtick.color'] = '#FF5533'

mlp.rcParams['ytick.major.size'] = 20
mlp.rcParams['ytick.major.width'] = 5
mlp.rcParams['ytick.labelsize'] = 20
mlp.rcParams['ytick.color'] = '#FF5533'

mlp.rcParams['axes.labelsize'] = 20
mlp.rcParams['axes.titlesize'] = 20
mlp.rcParams['axes.titlecolor'] = '#00B050'
mlp.rcParams['axes.labelcolor'] = '#00B050'

In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.options.display.max_columns = 500

df = pd.read_excel('processed_segmentation.xlsx')

In [24]:
df.head()

,Age,Work_Experience,Family_Size,Segmentation,Gender_Male,Ever_Married_Yes,Graduated_Yes,Profession_B,Profession_C,Profession_D,Spending_Score_High,Spending_Score_Low,Var_1_B,Var_1_C,Var_1_D
0,22,1.000000,4.0,D,1,0,0,0.075826,0.109610,0.734985,0,1,0.213039,0.100092,0.381084
1,38,2.641663,3.0,A,0,1,1,0.270386,0.107296,0.251788,0,0,0.213039,0.100092,0.381084
2,67,1.000000,1.0,B,0,1,1,0.270386,0.107296,0.251788,0,1,0.234287,0.285472,0.248400
3,67,0.000000,2.0,B,1,1,1,0.253612,0.224719,0.205457,1,0,0.234287,0.285472,0.248400
4,40,2.641663,6.0,A,0,1,1,0.232877,0.155954,0.226554,1,0,0.234287,0.285472,0.248400


In [11]:
### Отделим таргеты

X = df.drop('Segmentation', axis=1)
Y = df['Segmentation']

In [12]:
### Центрируем данные

X = X.subtract(X.mean())

X.head()

TypeError: can only concatenate str (not "int") to str

In [ ]:
### Проверка

X.sum()

### PCA Анализ

In [ ]:
### Разложим матрицу Х на 2 главные компоненты

from sklearn.decomposition import PCA

pca = PCA(n_components=2)

PCA_dataset = pca.fit_transform(X)

PCA_dataset = pd.DataFrame(PCA_dataset, columns=['PCA_1', 'PCA_2'])

PCA_dataset.head()

In [ ]:
### Что содержится в новых признаках?


first_component_corr = X.corrwith(PCA_dataset.PCA_1)
second_component_corr = X.corrwith(PCA_dataset.PCA_2)


corrs = pd.concat((first_component_corr, second_component_corr),
                  axis=1)

corrs.columns = ['PCA_1', 'PCA_2']

corrs

In [ ]:
import seaborn as sns

fig = plt.figure()

fig.set_size_inches(16, 10)

sns.heatmap(corrs,
            xticklabels=corrs.columns,
            yticklabels=corrs.index,
            cmap='BrBG',
            vmin=-1,
            vmax=1)

plt.show()

In [ ]:
### Сконкатим с таргетом

PCA_dataset = np.concatenate((PCA_dataset.values, Y.values.reshape(-1, 1)),
                              axis=1)

PCA_dataset = pd.DataFrame(PCA_dataset, columns=['PCA1', 'PCA2', 'SEGMENT'])

In [ ]:
PCA_dataset

In [ ]:
import seaborn as sns

fig = plt.figure()
fig.set_size_inches(16, 10)

sns.scatterplot(data=PCA_dataset, x="PCA1", y="PCA2", hue="SEGMENT")

In [ ]:
### Провернем все то же самое для 3 компонент!

pca_3d = PCA(n_components=3)

pca_3d.fit(X)

PCA_dataset_3d = pca_3d.transform(X)

PCA_dataset_3d = pd.DataFrame(PCA_dataset_3d, columns=['PCA_1', 'PCA_2', 'PCA_3'])

PCA_dataset_3d.head()

In [ ]:
### Что содержится в новых признаках?

first_component_corr_3d = X.corrwith(PCA_dataset_3d.PCA_1)
second_component_corr_3d = X.corrwith(PCA_dataset_3d.PCA_2)
third_component_corr_3d = X.corrwith(PCA_dataset_3d.PCA_3)

corrs_3d = pd.concat((first_component_corr_3d,
                      second_component_corr_3d,
                      third_component_corr_3d), axis=1)

corrs_3d.columns = ['PCA_1', 'PCA_2', 'PCA_3']

corrs_3d

In [ ]:
fig = plt.figure()

fig.set_size_inches(16, 10)

sns.heatmap(corrs_3d,
            xticklabels=corrs.columns,
            yticklabels=corrs.index,
            cmap='BrBG',
            vmin=-1,
            vmax=1)

plt.show()

In [ ]:
### Сконкатим с таргетом

PCA_dataset_3d = np.concatenate((PCA_dataset_3d.values, Y.values.reshape(-1, 1)),
                                 axis=1)

PCA_dataset_3d = pd.DataFrame(PCA_dataset_3d, columns=['PCA_1', 'PCA_2', 'PCA_3', 'SEGMENT'])

In [ ]:
fig = plt.figure()
fig.set_size_inches(16, 10)

ax = plt.axes(projection='3d')

colors = PCA_dataset_3d['SEGMENT'].replace(['A', 'B', 'C', 'D'],
                                            ['orange', 'green', 'red', 'blue'])

ax.scatter3D(PCA_dataset_3d['PCA_1'],
             PCA_dataset_3d['PCA_2'],
             PCA_dataset_3d['PCA_3'],
             c=colors)

In [ ]:
Y

In [ ]:
### Замерим качество и скорость работы такой модели

from sklearn.pipeline import Pipeline
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import SGDClassifier
from sklearn.preprocessing import StandardScaler

import datetime

pipe = Pipeline([('scaler', StandardScaler()),
                 ('classifier', OneVsRestClassifier(SGDClassifier()))])

begin_time = datetime.datetime.now()

pipe.fit(X, Y)

print(datetime.datetime.now() - begin_time)
print(pipe.score(X, Y))

In [ ]:
### Замерим качество и скорость работы такой модели

pipe = Pipeline([('scaler', StandardScaler()),
                 ('one_vs_all', OneVsRestClassifier(SGDClassifier()))])

begin_time = datetime.datetime.now()

pipe.fit(PCA_dataset_3d.drop('SEGMENT', axis=1), Y)

print(datetime.datetime.now() - begin_time)
print(pipe.score(PCA_dataset_3d.drop('SEGMENT', axis=1), Y))

In [ ]:
### Замерим качество для разного количества компонент

score_dict = {}
time_dict = {}

for power in range(1, 11):

    pca_dataset = PCA(n_components=power).fit_transform(X)

    begin_time = datetime.datetime.now()

    pipe.fit(pca_dataset, Y)

    time_dict[power] = (datetime.datetime.now() - begin_time).microseconds
    score_dict[power] = pipe.score(pca_dataset, Y)




In [ ]:
score_dict

In [ ]:
### Изобразим обучающую кривую

fig = plt.figure()
fig.set_size_inches(16, 10)

plt.plot(list(score_dict.keys()), list(score_dict.values()))

plt.show()


In [ ]:
### Изобразим обучающую кривую по времени

fig = plt.figure()
fig.set_size_inches(16, 10)

plt.plot(list(time_dict.keys()), list(time_dict.values()))

plt.show()

### TSNE

In [ ]:
### Здесь данные центрировать необходимости нет

X = df.drop('Segmentation', axis=1)
Y = df['Segmentation']

In [ ]:
### Произведем T-SNE преобразование

from sklearn.manifold import TSNE

X_tsne = TSNE(n_components=2).fit_transform(X)

X_tsne

In [ ]:
### Преобразуем в pd.DataFrame

X_tsne = np.concatenate((X_tsne, Y.values.reshape(-1, 1)),
                               axis=1)

X_tsne = pd.DataFrame(X_tsne, columns=['Tsne_1st_component',
                                       'Tsne_2nd_component',
                                       'SEGMENT'])

In [ ]:
import seaborn as sns

fig = plt.figure()
fig.set_size_inches(16, 10)

sns.scatterplot(data=X_tsne, x="Tsne_1st_component",
                y="Tsne_2nd_component",
                hue="SEGMENT")

In [ ]:
### Произведем T-SNE преобразование 3D

from sklearn.manifold import TSNE

X_tsne_3d = TSNE(n_components=3).fit_transform(X)

X_tsne_3d

In [ ]:
### Преобразуем в pd.DataFrame

X_tsne_3d = np.concatenate((X_tsne_3d, Y.values.reshape(-1, 1)),
                               axis=1)

X_tsne_3d = pd.DataFrame(X_tsne_3d, columns=['Tsne_1st_component',
                                             'Tsne_2nd_component',
                                             'Tsne_3rd_component',
                                             'SEGMENT'])

In [ ]:
fig = plt.figure()
fig.set_size_inches(16, 10)

ax = plt.axes(projection='3d')

colors = X_tsne_3d['SEGMENT'].replace(['A', 'B', 'C', 'D'],
                                      ['orange', 'green', 'red', 'blue'])

ax.scatter3D(X_tsne_3d['Tsne_1st_component'],
             X_tsne_3d['Tsne_2nd_component'],
             X_tsne_3d['Tsne_3rd_component'],
             c=colors)

In [ ]:
### Замерим качество и скорость работы такой модели

begin_time = datetime.datetime.now()

pipe = Pipeline([('scaler', StandardScaler()),
                 ('one_vs_all', OneVsRestClassifier(SGDClassifier()))])

pipe.fit(X_tsne_3d.drop('SEGMENT', axis=1), Y)

print(datetime.datetime.now() - begin_time)
print(pipe.score(X_tsne_3d.drop('SEGMENT', axis=1), Y))